<a href="https://colab.research.google.com/github/madelsu/MOSAIC-Agentic-Severity-Phenotyping/blob/main/Gold_Standards(Clinical_Benchmarks)/Ground_Truth_FINAL_v03_05_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — Define GT Scoring Functions                    ║
# ╚══════════════════════════════════════════════════════════╝
#
# INPUT  : nothing (defines functions and concept ID dictionaries)
#
# OUTPUT : Three reusable functions:
#   slice_window()             — filter a df to a specific time window
#   compute_dcsi()             — Young et al. DCSI-based GT scoring
#   extract_cooper_dimensions()— Cooper et al. 7-dimension GT scoring
#
# REFERENCE SYSTEMS:
#   Young et al.  : DCSI (Diabetes Complications Severity Index)
#                   7 domains scored 0-2, summed → 4 tiers
#   Cooper et al. : JBI 2025, 7 clinical dimensions → 4 tiers
#                   Uses green-rated (≥8/10) phenotypes only

# ── Window slicer ─────────────────────────────────────────────
def slice_window(df, date_col, start_col, end_col, ids=None):
    """Keep rows within a patient-specific time window."""
    d = df[df["PATIENT"].isin(ids)].copy() if ids else df.copy()
    return d[(d[date_col] >= d[start_col]) & (d[date_col] <= d[end_col])]

# ══════════════════════════════════════════════════════════════
# YOUNG ET AL. — DCSI CONCEPT IDs (SNOMED codes)
# ══════════════════════════════════════════════════════════════
DCSI = {
    "ret_2": [97331000119101, 1501000119109],        # Proliferative / Macular edema
    "ret_1": [422034002, 1551000119108, 157141000119108],  # Background / NPDR
    "neph_2": [127013003],                           # Diabetic renal disease
    "neph_1": [90781000119102],                      # Microalbuminuria
    "neu_2":  [],                                    # (not in Synthea)
    "neu_1":  [60951000119105, 368581000119106],     # Polyneuropathy / Peripheral
    "cbv_2":  [230690007],                           # Stroke
    "cbv_1":  [266257000],                           # TIA
    "cvd_2":  [22298006, 233829000],                 # MI / Heart attack
    "cvd_1":  [194828000],                           # Angina
    "pvd_2":  [],                                    # (not in Synthea)
    "pvd_1":  [698754002],                           # Peripheral arterial disease
    "met_2":  [420422005],                           # DKA
    "met_1":  [237636009],                           # Hypoglycaemia
}

# Lab LOINC codes for Young scoring
LAB_IDS_YOUNG = {
    "creat": "2160-0",    # Serum creatinine (max = worst)
    "uacr":  "14959-1",   # Microalbumin/creatinine ratio (max)
    "hba1c": "4548-4",    # HbA1c (max)
}

def compute_dcsi(con_slice, meas_slice, patient_ids, prefix):
    """
    Score DCSI for each patient using conditions and measurements
    from a pre-filtered time window slice.

    Parameters
    ----------
    con_slice    : conditions df filtered to the window
    meas_slice   : observations df filtered to the window
    patient_ids  : list of patient UUIDs to score
    prefix       : string prefix for output columns (e.g. 't5')

    Returns
    -------
    DataFrame with one row per patient and columns:
      {prefix}_ret, _neph, _neu, _cbv, _cvd, _pvd, _met  → domain scores (0-2)
      {prefix}_dcsi                                        → total score
      {prefix}_creat, _uacr, _hba1c                       → lab values
      {prefix}_young_tier                                  → 4-tier classification
      {prefix}_young_evidence                              → human-readable evidence
    """
    def pts(cids):
        if not cids: return set()
        m = con_slice["CODE"].dropna().astype(float).isin(cids)
        return set(con_slice[m]["PATIENT"].unique())

    domain_sets = {k: pts(v) for k, v in DCSI.items()}

    def lab_stat(concept_id, agg="min"):
        meas_slice["VALUE_NUM"] = pd.to_numeric(meas_slice["VALUE"], errors="coerce")
        sub = (meas_slice[meas_slice["CODE"].astype(str) == str(concept_id)]
               .dropna(subset=["VALUE_NUM"]).query("VALUE_NUM > 0"))
        return sub.groupby("PATIENT")["VALUE_NUM"].agg(agg) if len(sub) else pd.Series(dtype=float)

    creat_s = lab_stat(LAB_IDS_YOUNG["creat"], "max")
    uacr_s  = lab_stat(LAB_IDS_YOUNG["uacr"],  "max")
    hba1c_s = lab_stat(LAB_IDS_YOUNG["hba1c"], "max")

    rows = []
    for pid in patient_ids:
        ret = 2 if pid in domain_sets["ret_2"] else 1 if pid in domain_sets["ret_1"] else 0
        creat = creat_s.get(pid, np.nan)
        uacr  = uacr_s.get(pid, np.nan)
        neph  = max(
            2 if pid in domain_sets["neph_2"] else 1 if pid in domain_sets["neph_1"] else 0,
            2 if pd.notna(creat) and creat > 2.0 else 1 if pd.notna(creat) and creat >= 1.5 else 0,
            1 if pd.notna(uacr)  and uacr  >= 30  else 0,
        )
        neu = 2 if pid in domain_sets["neu_2"] else 1 if pid in domain_sets["neu_1"] else 0
        cbv = 2 if pid in domain_sets["cbv_2"] else 1 if pid in domain_sets["cbv_1"] else 0
        cvd = 2 if pid in domain_sets["cvd_2"] else 1 if pid in domain_sets["cvd_1"] else 0
        pvd = 2 if pid in domain_sets["pvd_2"] else 1 if pid in domain_sets["pvd_1"] else 0
        met = 2 if pid in domain_sets["met_2"] else 1 if pid in domain_sets["met_1"] else 0
        total = ret + neph + neu + cbv + cvd + pvd + met
        hba1c = hba1c_s.get(pid, np.nan)

        ev = []
        if ret  > 0: ev.append(f"retinopathy={ret}")
        if neph > 0: ev.append(f"nephropathy={neph}")
        if neu  > 0: ev.append(f"neuropathy={neu}")
        if cbv  > 0: ev.append(f"cerebrovascular={cbv}")
        if cvd  > 0: ev.append(f"cardiovascular={cvd}")
        if pvd  > 0: ev.append(f"peripheral_vasc={pvd}")
        if met  > 0: ev.append(f"metabolic={met}")
        if pd.notna(creat): ev.append(f"creatinine={creat:.2f}")
        if pd.notna(uacr):  ev.append(f"uacr={uacr:.1f}")
        if pd.notna(hba1c): ev.append(f"hba1c={hba1c:.1f}%")

        rows.append({
            "PATIENT":              pid,
            f"{prefix}_ret":        ret,
            f"{prefix}_neph":       neph,
            f"{prefix}_neu":        neu,
            f"{prefix}_cbv":        cbv,
            f"{prefix}_cvd":        cvd,
            f"{prefix}_pvd":        pvd,
            f"{prefix}_met":        met,
            f"{prefix}_dcsi":       total,
            f"{prefix}_creat":      round(creat, 2) if pd.notna(creat) else None,
            f"{prefix}_uacr":       round(uacr,  1) if pd.notna(uacr)  else None,
            f"{prefix}_hba1c":      round(hba1c, 1) if pd.notna(hba1c) else None,
            f"{prefix}_young_evidence": "; ".join(ev) if ev else "no complications detected",
        })

    df = pd.DataFrame(rows)
    df[f"{prefix}_young_tier"] = pd.cut(
        df[f"{prefix}_dcsi"], bins=[-1, 0, 1, 3, 13],
        labels=["Baseline T2D", "Mild Complications",
                "Moderate Complications", "Advanced/Critical"]
    ).astype(str)
    return df

# ══════════════════════════════════════════════════════════════
# COOPER ET AL. — CONCEPT IDs & SCORING FUNCTION
# ══════════════════════════════════════════════════════════════
COOPER_COMPLICATIONS = {
    "Nephropathy/CKD":  [127013003, 90781000119102, 46177005, 709044004],
    "Retinopathy":      [422034002, 1551000119108, 157141000119108, 97331000119101, 1501000119109],
    "Neuropathy":       [60951000119105, 368581000119106, 199049005],
    "Foot/Amputation":  [37130007, 129331000119104],
    "Stroke/CBV":       [230690007, 266257000],
    "Coronary/MI":      [22298006, 233829000, 194828000],
    "Heart_failure":    [84114007],
    "Peripheral_vasc":  [698754002],
}
RET_IDS   = {3: [97331000119101, 1501000119109], 2: [1551000119108, 157141000119108], 1: [422034002]}
RET_NAMES = {0: "None", 1: "Background", 2: "NPDR", 3: "Proliferative/Edema"}
LAB_IDS_COOPER = {"egfr": "33914-3", "uacr": "14959-1", "hba1c": "4548-4"}
DRUG_CLASSES    = {
    "Metformin":    [860975, 860978, 860981, 860996, 861004, 861006, 861010],
    "Insulin":      [847230, 847207, 847209, 847257, 106892],
    "Sulfonylurea": [316049, 197805],
}

def extract_cooper_dimensions(con_slice, meas_slice, med_slice, patient_ids, prefix):
    """
    Score Cooper et al. 7-dimension phenotype for each patient.

    Parameters
    ----------
    con_slice    : conditions df filtered to the window
    meas_slice   : observations df filtered to the window
    med_slice    : medications df filtered to the window
    patient_ids  : list of patient UUIDs to score
    prefix       : string prefix for output columns (e.g. 't5')

    Returns
    -------
    DataFrame with one row per patient and columns:
      {prefix}_comp_count, _uacr_cat, _ret_stage, _n_meds,
      {prefix}_insulin, _has_foot, _egfr_min, _hba1c_max,
      {prefix}_macro_flag, _micro_flag
      {prefix}_cooper_tier     → 4-tier classification
      {prefix}_cooper_evidence → human-readable evidence string
    """
    def pts_cond(cids):
        if not cids: return set()
        m = con_slice["CODE"].dropna().astype(float).isin(cids)
        return set(con_slice[m]["PATIENT"].unique())

    def pts_med(cids):
        if not cids: return set()
        m = med_slice["CODE"].dropna().astype(float).isin(cids)
        return set(med_slice[m]["PATIENT"].unique())

    def lab_stat(loinc, agg):
        meas_slice["VALUE_NUM"] = pd.to_numeric(meas_slice["VALUE"], errors="coerce")
        sub = (meas_slice[meas_slice["CODE"].astype(str) == str(loinc)]
               .dropna(subset=["VALUE_NUM"]).query("VALUE_NUM > 0"))
        return sub.groupby("PATIENT")["VALUE_NUM"].agg(agg) if len(sub) else pd.Series(dtype=float)

    uacr_max  = lab_stat(LAB_IDS_COOPER["uacr"],  "max")
    egfr_min  = lab_stat(LAB_IDS_COOPER["egfr"],  "min")
    hba1c_max = lab_stat(LAB_IDS_COOPER["hba1c"], "max")
    cond_sets = {k: pts_cond(v) for k, v in COOPER_COMPLICATIONS.items()}
    ret_sets  = {k: pts_cond(v) for k, v in RET_IDS.items()}
    med_sets  = {k: pts_med(v)  for k, v in DRUG_CLASSES.items()}

    rows = []
    for pid in patient_ids:
        comps   = sum(1 for s in cond_sets.values() if pid in s)
        uacr    = uacr_max.get(pid, np.nan)
        egfr    = egfr_min.get(pid,  np.nan)
        hba1c   = hba1c_max.get(pid, np.nan)
        uacr_cat = ("Unknown" if pd.isna(uacr) else
                    "Normal" if uacr < 30 else
                    "Microalbuminuria" if uacr < 300 else "Macroalbuminuria")
        ret_stage    = (3 if pid in ret_sets[3] else 2 if pid in ret_sets[2] else
                        1 if pid in ret_sets[1] else 0)
        n_meds       = sum(1 for s in med_sets.values() if pid in s)
        on_insulin   = int(pid in med_sets["Insulin"])
        has_foot     = int(pid in cond_sets["Foot/Amputation"])
        has_macro    = int(pid in cond_sets["Stroke/CBV"] or
                           pid in cond_sets["Coronary/MI"] or
                           pid in cond_sets["Heart_failure"] or
                           pid in cond_sets["Peripheral_vasc"])
        has_micro    = int(pid in cond_sets["Nephropathy/CKD"] or
                           pid in cond_sets["Retinopathy"] or
                           pid in cond_sets["Neuropathy"] or
                           uacr_cat in ["Microalbuminuria", "Macroalbuminuria"] or
                           (pd.notna(egfr) and egfr < 60))

        if comps >= 3 or has_foot or ret_stage == 3 or (pd.notna(egfr) and egfr < 30):
            tier = "Advanced/Critical"
        elif has_macro or has_micro:
            tier = "Moderate"
        elif (pd.notna(hba1c) and hba1c > 8.0) or n_meds > 1:
            tier = "Mild"
        else:
            tier = "Baseline T2D"

        ev = [f"comp_count={comps}", f"uacr={uacr_cat}", f"retinopathy={RET_NAMES[ret_stage]}"]
        if on_insulin:           ev.append("insulin=True")
        if has_foot:             ev.append("foot_ulcer=True")
        if pd.notna(egfr):       ev.append(f"egfr={egfr:.1f}")
        if pd.notna(hba1c):      ev.append(f"hba1c={hba1c:.1f}%")
        if has_macro:            ev.append("macrovascular=True")
        if has_micro:            ev.append("microvascular=True")

        rows.append({
            "PATIENT":                    pid,
            f"{prefix}_comp_count":       comps,
            f"{prefix}_uacr_cat":         uacr_cat,
            f"{prefix}_ret_stage":        RET_NAMES[ret_stage],
            f"{prefix}_n_meds":           n_meds,
            f"{prefix}_insulin":          on_insulin,
            f"{prefix}_has_foot":         has_foot,
            f"{prefix}_egfr_min":         round(egfr,   1) if pd.notna(egfr)   else None,
            f"{prefix}_hba1c_max":        round(hba1c,  1) if pd.notna(hba1c)  else None,
            f"{prefix}_macro_flag":       has_macro,
            f"{prefix}_micro_flag":       has_micro,
            f"{prefix}_cooper_tier":      tier,
            f"{prefix}_cooper_evidence":  "; ".join(ev),
        })
    return pd.DataFrame(rows)

print("✅ GT scoring functions defined:")
print("   slice_window()")
print("   compute_dcsi()               — Young et al. DCSI")
print("   extract_cooper_dimensions()  — Cooper et al. 7-dim")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Score All 4 Time Windows (Young + Cooper GT)   ║
# ╚══════════════════════════════════════════════════════════╝
#
# INPUT  : cohort, con_all, obs_all, med_all (from Cells 3-4)
#          slice_window, compute_dcsi, extract_cooper_dimensions (Cell 5)
#
# OUTPUT : young_dfs  — dict {prefix: DataFrame} with Young GT scores
#          cooper_dfs — dict {prefix: DataFrame} with Cooper GT scores
#          cohort     — updated with all GT scores merged in
#
# WINDOWS SCORED:
#   td  — At diagnosis         (all 35,875 patients)
#   t0  — At first treatment   (all 35,875 patients)
#   t5  — 5-year reclassification (35,368 eligible patients)
#   t10 — 10-year reclassification (31,263 eligible patients)

eligible_ids = set(cohort["PATIENT"])
ids_5        = set(cohort.loc[cohort["eligible_reclass_5"]  == 1, "PATIENT"])
ids_10       = set(cohort.loc[cohort["eligible_reclass_10"] == 1, "PATIENT"])

# Window configuration: (conditions_date, obs_date, med_date, start_col, end_col, patient_ids)
windows_map = {
    "td":  ("START", "DATE", "START", "td_cov_start",  "td_cov_end",  eligible_ids),
    "t0":  ("START", "DATE", "START", "t0_cov_start",  "t0_cov_end",  eligible_ids),
    "t5":  ("START", "DATE", "START", "t5_cov_start",  "t5_cov_end",  ids_5),
    "t10": ("START", "DATE", "START", "t10_cov_start", "t10_cov_end", ids_10),
}

young_dfs  = {}
cooper_dfs = {}

for prefix, (con_dc, obs_dc, med_dc, start_col, end_col, ids) in windows_map.items():
    print(f"\n{'='*55}")
    print(f"  Scoring {prefix.upper()} — {len(ids):,} patients")
    print(f"{'='*55}")

    con_sl  = slice_window(con_all,  con_dc, start_col, end_col, ids)
    obs_sl  = slice_window(obs_all,  obs_dc, start_col, end_col, ids)
    med_sl  = slice_window(med_all,  med_dc, start_col, end_col, ids)

    print(f"  Sliced: conditions={len(con_sl):,} | obs={len(obs_sl):,} | meds={len(med_sl):,}")

    print(f"  Young GT...")
    y = compute_dcsi(con_sl, obs_sl, list(ids), prefix)
    young_dfs[prefix] = y
    print(y[f"{prefix}_young_tier"].value_counts().to_string())

    print(f"  Cooper GT...")
    c = extract_cooper_dimensions(con_sl, obs_sl, med_sl, list(ids), prefix)
    cooper_dfs[prefix] = c
    print(c[f"{prefix}_cooper_tier"].value_counts().to_string())

# Merge all GT scores into cohort
print("\nMerging GT scores into cohort...")
for prefix in ["td", "t0", "t5", "t10"]:
    cohort = cohort.merge(young_dfs[prefix],  on="PATIENT", how="left")
    cohort = cohort.merge(cooper_dfs[prefix], on="PATIENT", how="left")

print(f"\n✅ All GT scores computed and merged!")
print(f"   cohort shape: {cohort.shape}")